# Exploratory Data Analysis and Data Visualisations. 

## This notebook focuses on data visualisation and exploratory data analysis (EDA) of all the images we are going to use to train AquaVision. 
It focuses on:
- Dataset composition. We want to see the total number of images, number of input species, image formats and dimensions, dataset structure and organisation. This allows us to establlsh the size and structure of our dataset.
-  Class distribution and imbalance. I will attempt to investigate the number of images per species, most and least represented species, distribution of images across classes and the degree of class imabalance. 
- Image Characteristics and Visual Quality. I will look at Image resoultion and aspect ratios, colour distributions, backgrounds, lighting and fish orientation, image quality and corrupt files. 
- Data Integrity and reliability. I will assess whether or not the dataset has missing or invalid labels, duplicate images, potentially mislabeled samples, unreadable images. 
- Raw VS Cropped Dataset Comparison. I will attempt to assess whether or not cropping will improve the dataset for fish species classification. 

In short, this notebook is aimed at understanding the data, identifying challenges, and informing modelling decsions, and will act as the foundation for our data cleaning notebook as well. 

### Imports

In [1]:
from pathlib import Path # Navigate directories and manage file paths
import os # Interact with the operating system and inspect file directory properties
import hashlib # Used to generate hashes to identify identical image files
import numpy as np # Used to perform numerical operations and calculate statistical measures
import pandas as pd # Organise dataset metadata and species counts into structures DataFrames
import matplotlib.pyplot as plt # Create charts, plots, and figures for exploring the dataset
import seaborn as sns # Create statistical visualisations, including species distribution plots
from PIL import Image # Open images, inspect their dimensions, formats, and colour moddes
from collections import Counter # Count occurences of species labels and other categorical values
from IPython.display import display, Markdown # Display formatted tables, and analysis results


### Visualisation Configurations

In [2]:
sns.set_theme(style="whitegrid", context="notebook") # Apply consistent styling to statistical plots 
plt.rcParams["figure.figsize"] = (12, 6) # Set the default figure size to 12 × 6 inches
plt.rcParams["figure.dpi"] = 120 # Set the resolution for figures displayed in the notebook
plt.rcParams["savefig.dpi"] = 300 # Set high resolution for exported figures
plt.rcParams["axes.titlesize"] = 14 # Set the default font size for chart titles
plt.rcParams["axes.labelsize"] = 11 # Set the default font size for axis labels
plt.rcParams["font.family"] = "sans-serif" # Use a clean sans-serif font for visual consistency.

### General Configurations

In [3]:
RANDOM_STATE = 42 # Provide a reproducible random seed for any randomised analysis

IMAGE_EXTENSIONS = { 
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff" 
    } # Define the image formats to recognise when scanning the dataset

### Enviornment Verification

In [4]:
print("AquaVision EDA environment initialised.") 
print(f"NumPy version: {np.__version__}") 
print(f"Pandas version: {pd.__version__}")
print(f"Seaborn version: {sns.__version__}") 
print(f"Matplotlib version: {plt.matplotlib.__version__}") 
print("All imports and configurations completed.")

AquaVision EDA environment initialised.
NumPy version: 2.3.5
Pandas version: 2.3.3
Seaborn version: 0.13.2
Matplotlib version: 3.10.7
All imports and configurations completed.


## Dataset overview

### Dataset Structure

In [5]:
# Define the dataset path
DATASET_PATH = Path("C:\\Users\\user\\OneDrive\\Documents\\B.Sc in AI\\third year everything\\3.1\\Deep Learning\\Aquavision\\data\\data_paper_dataset")

# Verify that the directory exists 
if DATASET_PATH.exists() and DATASET_PATH.is_dir():
    print("Dataset directory found!")
    print(f"Dataset location: {DATASET_PATH.resolve()}")
else:
    print("Dataset directory not found. Please check DATASET_PATH")                        

Dataset directory found!
Dataset location: C:\Users\user\OneDrive\Documents\B.Sc in AI\third year everything\3.1\Deep Learning\Aquavision\data\data_paper_dataset


In [9]:
# List the files and folders in the dataset's root directory
dataset_contents = sorted(DATASET_PATH.iterdir())

print(f"Total items in dataset root: {len(dataset_contents)}\n")

for item in dataset_contents:
    item_type = "Directory" if item.is_dir() else "File"
    print(f"{item_type}: {item.name}")

Total items in dataset root: 160

Directory: _calcareous
Directory: _coral_hard
Directory: _coral_soft
Directory: _dark
Directory: _macroalgae
Directory: _sand
Directory: _water_blue
Directory: _water_surface
Directory: Abudefduf_sexfasciatus
Directory: Abudefduf_sparoides
Directory: Abudefduf_vaigiensis
Directory: Acanthurus_leucosternon
Directory: Acanthurus_lineatus
Directory: Acanthurus_nigricauda
Directory: Acanthurus_tennenti
Directory: Acanthurus_thompsoni
Directory: Aethaloperca_rogaa
Directory: Amanses_scopas
Directory: Amblyglyphidodon_indicus
Directory: Anampses_caeruleopunctatus
Directory: Anampses_lineatus
Directory: Anampses_meleagrides_female
Directory: Anampses_twistii
Directory: Apolemichthys_trimaculatus
Directory: Arothron_nigropunctatus
Directory: Aulostomus_chinensis
Directory: Balistapus_undulatus
Directory: Bodianus_diana
Directory: Caesio_lunaris
Directory: Caesio_xanthonota
Directory: Canthigaster_bennetti
Directory: Canthigaster_valentini
Directory: Caranx_mel

### Dataset Category Identification